In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn
%matplotlib inline

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error,root_mean_squared_error

In [3]:
df=pd.read_csv('../data/healthcare_cleaned.csv')

In [4]:
drop_cols=['patient_id','treatment_cost','mortality_risk','diagnosis_category','length_of_stay','readmission_30day']

In [5]:
X=df.drop(drop_cols,axis=1)
y=df['length_of_stay']

In [6]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
cat_cols=[col for col in X.columns if X[col].dtypes=='str']
num_cols=[col for col in X.columns if col not in cat_cols]
preprocessor=ColumnTransformer(transformers=[
    ('num',StandardScaler(),num_cols),
    ('cat',OneHotEncoder(),cat_cols)
])
linear_reg_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',LinearRegression())
])
linear_reg_pipeline.fit(X_train,y_train)
y_pred=linear_reg_pipeline.predict(X_test)
print(r2_score(y_test,y_pred))
print(root_mean_squared_error(y_test,y_pred))

0.44325908387419255
1.4931060890347054


In [8]:
y_log=np.log1p(y)

In [9]:
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

In [10]:
linear_reg_pipeline.fit(X_train,y_train_log)
y_pred=linear_reg_pipeline.predict(X_test)
y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test_log)
print(r2_score(y_test_actual, y_pred_actual))
print(root_mean_squared_error(y_test_actual, y_pred_actual))

0.43818672367046363
1.4998923692260777


observations:
1) r2 is score weak which 0.44 , and rmse is nearly avg of 1.5 days
2) in the ede , when target is having of skew of 0.87, we found log transform does not help , so pushing raw data for complex models
3) as many features not contributing or possess weak signals , skipping regularization of linear models as data is non-linear.



In [11]:
from sklearn.ensemble import RandomForestRegressor
random_forest_reg_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',RandomForestRegressor())
])
random_forest_reg_pipeline.fit(X_train,y_train)
y_pred=random_forest_reg_pipeline.predict(X_test)
print(r2_score(y_test,y_pred))
print(root_mean_squared_error(y_test,y_pred))

0.4109952710954631
1.535760479534488


In [12]:
from sklearn.model_selection import GridSearchCV
params={
    'model__n_estimators':[50,100,150],
    'model__max_depth':[3,5,7],
    'model__min_samples_split':[3,5,8,10]
}
grid_model_rf=GridSearchCV(param_grid=params,estimator=random_forest_reg_pipeline,n_jobs=-1,cv=5,scoring='r2')
grid_model_rf.fit(X_train,y_train)
y_pred=grid_model_rf.predict(X_test)
print(r2_score(y_test,y_pred))

0.4524394444439168


In [13]:
print(root_mean_squared_error(y_test,y_pred))

1.4807446576013317


In [14]:
print(grid_model_rf.best_params_)
print(grid_model_rf.best_score_)

{'model__max_depth': 7, 'model__min_samples_split': 8, 'model__n_estimators': 150}
0.4622725329294755


In [15]:
from xgboost import XGBRegressor
xgb_reg_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',XGBRegressor())
])
xgb_reg_pipeline.fit(X_train,y_train)
y_pred=xgb_reg_pipeline.predict(X_test)
print(r2_score(y_test,y_pred))
print(root_mean_squared_error(y_test,y_pred))

0.4172039039524885
1.5276448969396104


In [18]:
params={
    'model__n_estimators':[50,100,150],
    'model__max_depth':[3,5,7],
    'model__learning_rate':[0.01,0.1,0.3]
}
grid_model_xgb=GridSearchCV(param_grid=params,estimator=xgb_reg_pipeline,n_jobs=-1,cv=5,scoring='r2')
grid_model_xgb.fit(X_train,y_train)
y_pred=grid_model_xgb.predict(X_test)
print(r2_score(y_test,y_pred))
print(root_mean_squared_error(y_test,y_pred))

0.4571712687005538
1.4743327398990584


In [19]:
print(grid_model_xgb.best_params_)
print(grid_model_xgb.best_score_)

{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
0.4685210325763033


In [21]:
import joblib
joblib.dump(grid_model_xgb,'../models/LOS.pkl')

['../models/LOS.pkl']

Result : XGBoost best model R²=0.457, RMSE=1.47 — predictions off by ~1.5 days on average
